<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Rates, Bonds, and Credit

## Appendix A · Stochastic Processes and Brownian Motion

[Open in Google Colab](https://colab.research.google.com/github/yhilpisch/ratescode/blob/main/notebooks/appx_a_stochastic_processes_brownian_motion.ipynb)

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Learning goals

After working through this notebook, you will be able to:

- define a stochastic process, a filtration, and an adapted process in fixed-income language;
- build Brownian motion as the limit of a scaled random walk whose increments have standard deviation equal to the square root of the time step;
- explain why volatility scales with the square root of time, so a one-month shock is about one over the square root of twelve of the annual volatility;
- compute realized quadratic variation on a simulated path and connect it to the rule that Brownian quadratic variation over the interval from 0 to T equals T; and
- separate the drift term, which scales linearly with the horizon, from the diffusion term, which scales with its square root.

## Prepare the companion repository

This setup cell works with a local checkout or a fresh Google Colab runtime. In Colab it clones the public companion repository, then sets `ROOT` so the chapter code and frozen data snapshots are found without manual path edits.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/yhilpisch/ratescode.git"
candidates = [Path.cwd(), Path.cwd().parent,
              Path("/content/07_ratescode")]
ROOT = next((p.resolve() for p in candidates
             if (p / "code").is_dir()
             and (p / "data").is_dir()), None)
if ROOT is None:
    ROOT = Path("/content/07_ratescode")
    subprocess.run(["git", "clone", "--depth", "1",
                    REPO_URL, str(ROOT)], check=True)

packages = {"numpy": "numpy", "pandas": "pandas",
            "matplotlib": "matplotlib"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install",
                    *missing], check=True)
print(f"Repository: {ROOT}")
print(f"Data files: {len(list((ROOT / 'data').glob('*.csv')))}")


## Connect the appendix to a calculation

The appendix supplies the process language behind the short-rate models of Chapter 12, the forward-rate dynamics of Chapter 13, and the credit intensity models of Chapter 17: a conditional mean plus shocks whose scale depends on the horizon. The companion figure script `code/figures/appx_a_brownian_paths_scaling.py` draws representative Brownian paths over one year next to the square-root scaling rule. This notebook complements the derivation with a numerical experiment: it simulates Brownian increments on a fine grid, sums them into a path, and accumulates the squared increments into realized quadratic variation, the quantity behind the rule that the squared Brownian increment behaves like the time step in Appendix B.

In [ ]:
import pandas as pd
figure_scripts = sorted((ROOT / "code" / "figures").glob("appx_a_*.py"))
pd.DataFrame({"figure_script": [p.name for p in figure_scripts]})

## Review prompts

Answer these before using the appendix result in a chapter calculation:

- Which measure do the simulated draws come from, and does the application need the real-world measure or a pricing measure?
- What are the time units of the step size, and is the volatility quoted per year, per month, or per day?
- Is the process adapted to the filtration used in the application, so that no future information leaks into decisions made at time t?
- How fine is the partition in the quadratic-variation sum, and how far is the realized value from the theoretical limit T?
- Over your horizon, does the drift contribution or the diffusion contribution dominate the change in the rate?

## Mini-lab: Sum Brownian increments and measure quadratic variation

This lab isolates the two accumulation rules of the appendix on one simulated path. Standard normal draws are scaled by the square root of the time step, so each increment has variance equal to the step size, and the path is their running sum. Squaring the increments and summing them gives the realized quadratic variation. The simulation uses 1,000 steps over a horizon of T equals 1 under the real-world measure; both statistics are fixed-seed illustrations of the mechanism, not estimates.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
steps, T = 1_000, 1.0
dt = T / steps
z = rng.standard_normal(steps)  # standard normal draws
increments = np.sqrt(dt) * z  # Brownian increments on [0, 1]
path = np.r_[0.0, np.cumsum(increments)]  # random-walk sum
qv = np.sum(increments ** 2)  # realized quadratic variation
pd.Series({"terminal_value": path[-1],
           "quadratic_variation": qv, "horizon": T})

### What to observe

With seed 42, the simulated path ends at about -0.91 while the sum of squared increments is about 0.98, close to the theoretical quadratic variation of 1 over the unit horizon. The terminal value is random; the quadratic variation is not, and it tightens toward T as the grid is refined.

## Exercises

1. **Refine the grid.** Repeat the calculation for 100 and 10,000 steps and describe how the realized quadratic variation moves toward the horizon T. *Hint:* Keep the seed fixed so only the partition changes.
2. **Rescale the horizon.** Set T to 1/12 for monthly steps and confirm that both the typical terminal move and the quadratic variation scale with T. *Hint:* The increment standard deviation is the square root of the step size.
3. **Add drift.** Take an annual drift of 12 basis points and an annual volatility of 100 basis points, and compute the one-month drift and one-standard-deviation diffusion contributions. *Hint:* The drift contribution is about 1 basis point, the diffusion contribution about 28.9.
4. **Check the scaling table.** Reproduce the four-step random-walk table from the appendix with the given standard normal values and match each entry. *Hint:* Each increment is the square root of 0.25 times the draw.
5. **Connect to a model.** Explain which parts of the Vasicek short-rate model in Chapter 12 this experiment makes concrete. *Hint:* Identify the drift term and the diffusion term and their time scaling.

For more detail, see Appendix A of *Python & AI for Rates, Bonds, and Credit*.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">